In [1]:
# %pip install -q --upgrade pip
# %pip install -q --upgrade tensorflow tensorflow-datasets transformers accelerate evaluate
# Il est recommandé de redémarrer l'environnement d'exécution (Runtime -> Restart runtime) après cette installation.

In [2]:
# import transformers

# print(transformers.__version__)
# print(hasattr(transformers, "TFBertForSequenceClassification"))
# print(hasattr(transformers, "BertTokenizer"))

# %pip install -U transformers

import platform
import transformers
import tensorflow as tf
import tensorflow_datasets as tfds
from transformers import BertTokenizer, TFBertForSequenceClassification

print("Python version      :", platform.python_version())
print("TensorFlow version  :", tf.__version__)
print("GPU devices detected:", tf.config.list_physical_devices("GPU"))


c:\Users\meles\Documents\TTA_DI_BootCamp_Gilles-Chris_MAKE\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Python version      : 3.12.10
TensorFlow version  : 2.21.0
GPU devices detected: []


In [3]:
import tensorflow_datasets as tfds

print(tfds)
print("Version :", tfds.__version__)
print("load existe :", hasattr(tfds, "load"))
print(dir(tfds)[:20])

<module 'tensorflow_datasets' from 'c:\\Users\\meles\\Documents\\TTA_DI_BootCamp_Gilles-Chris_MAKE\\.venv\\Lib\\site-packages\\tensorflow_datasets\\__init__.py'>
Version : 4.9.10
load existe : True
['GenerateMode', 'ImageFolder', 'ReadConfig', 'Split', 'TranslateFolder', '__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', 'annotations', 'as_dataframe', 'as_numpy', 'audio']


In [4]:
import tensorflow_datasets as tfds

(ds_train, ds_test), ds_info = tfds.load(
    "imdb_reviews",
    split=["train", "test"],
    as_supervised=True,
    with_info=True
)
print(ds_info)

tfds.core.DatasetInfo(
    name='imdb_reviews',
    full_name='imdb_reviews/plain_text/1.0.0',
    description="""
    Large Movie Review Dataset. This is a dataset for binary sentiment
    classification containing substantially more data than previous benchmark
    datasets. We provide a set of 25,000 highly polar movie reviews for training,
    and 25,000 for testing. There is additional unlabeled data for use as well.
    """,
    config_description="""
    Plain text
    """,
    homepage='http://ai.stanford.edu/~amaas/data/sentiment/',
    data_dir='C:\\Users\\meles\\tensorflow_datasets\\imdb_reviews\\plain_text\\1.0.0',
    file_format=tfrecord,
    download_size=80.23 MiB,
    dataset_size=129.83 MiB,
    features=FeaturesDict({
        'label': ClassLabel(shape=(), dtype=int64, num_classes=2),
        'text': Text(shape=(), dtype=string),
    }),
    supervised_keys=('text', 'label'),
    disable_shuffling=False,
    nondeterministic_order=False,
    splits={
        'test': <

In [5]:
for text, label in ds_train.take(2):
    print("Label:", "Positive" if label.numpy() else "Negative")
    print(text.numpy().decode()[:250], "...\n")

Label: Negative
This was an absolutely terrible movie. Don't be lured in by Christopher Walken or Michael Ironside. Both are great actors, but this must simply be their worst role in history. Even their great acting could not redeem this movie's ridiculous storyline ...

Label: Negative
I have been known to fall asleep during films, but this is usually due to a combination of things including, really tired, being warm and comfortable on the sette and having just eaten a lot. However on this occasion I fell asleep because the film wa ...



In [6]:
MAX_LENGTH = 256   # trim or pad every review to 256 tokens so batches align
BATCH_SIZE = 16

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased", do_lower_case=True)
print("Tokenizer loaded:", tokenizer.name_or_path)

Tokenizer loaded: bert-base-uncased


In [7]:
MAX_LENGTH = 256
BATCH_SIZE = 16
AUTOTUNE = tf.data.AUTOTUNE

if 'tokenizer' not in globals():
    tokenizer = BertTokenizer.from_pretrained("bert-base-uncased", do_lower_case=True)
    print("Tokenizer loaded:", tokenizer.name_or_path)


def encode_example(text, label):
    def tokenize_text(text_value):
        if hasattr(text_value, "numpy"):
            text_value = text_value.numpy()
        if isinstance(text_value, bytes):
            text_value = text_value.decode("utf-8")

        encoded = tokenizer(
            text_value,
            add_special_tokens=True,
            max_length=MAX_LENGTH,
            padding="max_length",
            truncation=True,
            return_attention_mask=True,
            return_token_type_ids=True,
            return_tensors="tf",
        )

        return (
            tf.cast(encoded["input_ids"][0], tf.int32),
            tf.cast(encoded["attention_mask"][0], tf.int32),
            tf.cast(encoded["token_type_ids"][0], tf.int32),
        )

    input_ids, attention_mask, token_type_ids = tf.py_function(
        func=tokenize_text,
        inp=[text],
        Tout=[tf.int32, tf.int32, tf.int32],
    )

    input_ids.set_shape([MAX_LENGTH])
    attention_mask.set_shape([MAX_LENGTH])
    token_type_ids.set_shape([MAX_LENGTH])
    label.set_shape([])

    return (
        {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "token_type_ids": token_type_ids,
        },
        label,
    )


def prepare_dataset(dataset, training=False):
    if training:
        dataset = dataset.shuffle(10000, reshuffle_each_iteration=True)

    return (
        dataset
        .map(encode_example, num_parallel_calls=AUTOTUNE)
        .batch(BATCH_SIZE)
        .prefetch(AUTOTUNE)
    )

train_ds = prepare_dataset(ds_train, training=True)
test_ds = prepare_dataset(ds_test, training=False)


In [5]:
import tensorflow as tf
from transformers import TFBertForSequenceClassification

model = TFBertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2,
    use_safetensors=False,
)

optimizer = tf.keras.optimizers.Adam(learning_rate=2e-5, epsilon=1e-8)
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
metrics = [tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy")]

model.compile(optimizer=optimizer, loss=loss_fn, metrics=metrics)
model.summary()

All model checkpoint layers were used when initializing TFBertForSequenceClassification.

Some layers of TFBertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model: "tf_bert_for_sequence_classification_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 bert (TFBertMainLayer)      multiple                  109482240 
                                                                 
 dropout_75 (Dropout)        multiple                  0 (unused)
                                                                 
 classifier (Dense)          multiple                  1538      
                                                                 
Total params: 109483778 (417.65 MB)
Trainable params: 109483778 (417.65 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
from transformers import BertTokenizer

if isinstance(model, tuple):
    model = model[0]

if "MAX_LENGTH" not in globals():
    MAX_LENGTH = 256
if "BATCH_SIZE" not in globals():
    BATCH_SIZE = 16
if "AUTOTUNE" not in globals():
    AUTOTUNE = tf.data.AUTOTUNE

if "tokenizer" not in globals():
    tokenizer = BertTokenizer.from_pretrained("bert-base-uncased", do_lower_case=True)
    print("Tokenizer loaded:", tokenizer.name_or_path)

def encode_example(text, label):
    def tokenize_text(text_value):
        if hasattr(text_value, "numpy"):
            text_value = text_value.numpy()
        if isinstance(text_value, bytes):
            text_value = text_value.decode("utf-8")

        encoded = tokenizer(
            text_value,
            add_special_tokens=True,
            max_length=MAX_LENGTH,
            padding="max_length",
            truncation=True,
            return_attention_mask=True,
            return_token_type_ids=True,
            return_tensors="tf",
        )

        return (
            tf.cast(encoded["input_ids"][0], tf.int32),
            tf.cast(encoded["attention_mask"][0], tf.int32),
            tf.cast(encoded["token_type_ids"][0], tf.int32),
        )

    input_ids, attention_mask, token_type_ids = tf.py_function(
        func=tokenize_text,
        inp=[text],
        Tout=[tf.int32, tf.int32, tf.int32],
    )

    input_ids.set_shape([MAX_LENGTH])
    attention_mask.set_shape([MAX_LENGTH])
    token_type_ids.set_shape([MAX_LENGTH])
    label.set_shape([])

    return (
        {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "token_type_ids": token_type_ids,
        },
        label,
    )

def prepare_dataset(dataset, training=False):
    if training:
        dataset = dataset.shuffle(10000, reshuffle_each_iteration=True)

    return (
        dataset
        .map(encode_example, num_parallel_calls=AUTOTUNE)
        .batch(BATCH_SIZE)
        .prefetch(AUTOTUNE)
    )

if "ds_train" not in globals() or "ds_test" not in globals():
    (ds_train, ds_test), ds_info = tfds.load(
        "imdb_reviews",
        split=["train", "test"],
        as_supervised=True,
        with_info=True,
    )

if "train_ds" not in globals() or "test_ds" not in globals():
    train_ds = prepare_dataset(ds_train, training=True)
    test_ds = prepare_dataset(ds_test, training=False)

history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=2,
)

results = model.evaluate(test_ds)
print("Test loss:", results[0])
print("Test accuracy:", results[1])

def predict_sentiment(text):
    encoded = tokenizer(
        text,
        add_special_tokens=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        truncation=True,
        return_attention_mask=True,
        return_token_type_ids=True,
        return_tensors="tf",
    )

    logits = model(
        {
            "input_ids": encoded["input_ids"],
            "attention_mask": encoded["attention_mask"],
            "token_type_ids": encoded["token_type_ids"],
        },
        training=False,
    ).logits

    prediction = tf.argmax(logits, axis=-1).numpy()[0]
    return "positive" if prediction == 1 else "negative"

examples = [
    "I loved this movie, the acting was great and the plot was gripping.",
    "This was a terrible film. I do not recommend it to anyone."
]

for example in examples:
    print(example)
    print("Predicted sentiment:", predict_sentiment(example))
    print()


ValueError: prepare_dataset is not defined. Run the dataset preparation cell first.

In [1]:
if isinstance(model, tuple):
    model = model[0]

history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=2,
)

NameError: name 'model' is not defined